# 04 - Database Layer and Multi-table Relations

> **When to use**: When you have multiple interrelated tables and need to ensure foreign key integrity.
>
> **Core concept**: sqlseed auto-detects table dependencies, fills in topological order, and shares values across tables via SharedPool.

## Use Cases

- Multiple tables with FOREIGN KEY constraints → sqlseed auto-orders fills
- Related columns without declared FKs → explicit ColumnAssociation
- Different column names but need association (e.g., `department_id` → `id`) → ColumnAssociation explicit association
- Large data volume write performance optimization → Pragma three-level optimization

## What You Will Learn

- Dual adapter architecture (SQLAlchemyAdapter / RawSQLiteAdapter)
- Pragma three-level write optimization
- SharedPool cross-table value sharing
- ColumnAssociation explicit association
- BLOB / large text handling

See architecture.zh-CN.md §5

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Flow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| **→ 04** | **Database Layer and Multi-table Relations** | **Database + Core** | **01** |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Test Integration Patterns | Testing | 01 |

---
## Setup

Use Python 3.10+ and run this notebook from `examples/notebooks` in a repository checkout. Select a notebook kernel from the environment containing these packages:

```bash
python -m pip install 'sqlseed[mimesis]==0.2.4' 'sqlseed-cli==0.2.4' jupyterlab
```

For source development, install Core and CLI together as described in the [repository README](../../README.md). This notebook creates its own temporary database and cache. Run cells from top to bottom; the validation helpers raise on partial generation or failed CLI commands.


In [ ]:
from pathlib import Path

# Install the packages listed in Setup into the selected notebook kernel.
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
import os
import tempfile
from pathlib import Path
notebook_temp = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-04-")
work_dir = Path(notebook_temp.name)
os.environ["SQLSEED_CACHE_DIR"] = str(work_dir / "cache")
db_path = build(work_dir / "demo.db")

# Fail visibly if a generation only partially succeeds.
generation_checks = []
def check_result(result, expected_count):
    if result.errors or result.count != expected_count:
        raise RuntimeError(f"{result.table_name}: expected {expected_count}, wrote {result.count}; errors={result.errors}")
    generation_checks.append({"table": result.table_name, "count": result.count, "errors": list(result.errors)})
    print(f"Verified {result.table_name}: {result.count} rows; errors={result.errors}")
    return result

def check_results(results, config_path):
    config = sqlseed.load_config(str(config_path))
    expected = {table.name: table.count for table in config.tables}
    for result in results:
        check_result(result, expected[result.table_name])
    if {result.table_name for result in results} != set(expected):
        raise RuntimeError("Not every configured table produced a result")
    return results

def check_cli(result):
    if result.exit_code != 0:
        raise RuntimeError(result.output) from result.exception
    return result


# Populate base dependencies
with connect(str(db_path)) as orch:
    check_result(orch.fill_table("organizations", count=5, seed=42), 5)
    check_result(orch.fill_table("members", count=20, seed=42), 20)
    check_result(orch.fill_table("projects", count=10, seed=42), 10)
    check_result(orch.fill_table("tags", count=8, seed=42), 8)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

### 📍 Architecture Location

| Module | File | Core Class/Function |
|------|------|------------|
| Relation resolution | `src/sqlseed/core/relation.py` | `RelationResolver` |
| Database interface | `src/sqlseed/database/_protocol.py` | `DatabaseAdapter` |

> See architecture diagram: [§5 Database Layer Architecture](../../docs/architecture.zh-CN.md#5-数据库层架构)

## 1. See It in Action — Multi-table Relation Auto-ordering

Database has foreign key constraints? sqlseed auto-detects table dependencies and fills in topological order — **no need to specify order manually**:
This example appends to the initialized database. Multi-table filling is not an atomic reset; it does not clear child tables before parents.


In [ ]:
from sqlseed import fill_from_config
from sqlseed.config.loader import save_config
from sqlseed.config.models import GeneratorConfig, TableConfig

# Fill 5 tables at once — sqlseed auto-handles FK order
config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name='organizations', count=3),
        TableConfig(name='members', count=10),
        TableConfig(name='projects', count=5),
        TableConfig(name='tasks', count=20),
        TableConfig(name='tags', count=5),
    ]
)
config_path = (work_dir / '_fk_demo.yaml')
save_config(config, str(config_path))

results = check_results(fill_from_config(str(config_path)), str(config_path))
print(f"{'Table':<15s}  {'Rows':>6s}  {'Elapsed':>8s}  {'Speed':>10s}")
print('-' * 45)
for r in results:
    print(f"{r.table_name:<15s}  {r.count:>6d}  {r.elapsed:>7.3f}s  {r.rows_per_second:>8.0f} rows/s")

config_path.unlink(missing_ok=True)

Note the output order — `organizations` before `members`, `projects` before `tasks`. sqlseed determines the correct fill order via topological sort, ensuring foreign key references are valid.

Below we break down each mechanism of the database layer in detail.

## 2. Dual Adapter Architecture

sqlseed supports two database adapters:

| Adapter | Dependency | Default | Characteristics |
|--------|------|:----:|------|
| `SQLAlchemyAdapter` | SQLAlchemy | ✅ | Feature-rich, multi-DB support |
| `RawSQLiteAdapter` | None (stdlib) | - | Zero-dep, test-only |

sqlseed auto-selects: `SQLAlchemyAdapter` is the required default for production (multi-DB support); `RawSQLiteAdapter` is a test-only zero-dependency fallback.

In [ ]:
from sqlseed.database import SQLAlchemyAdapter, RawSQLiteAdapter

# SQLAlchemyAdapter is the required production adapter (multi-DB support).
# RawSQLiteAdapter is a test-only, zero-dependency fallback.
print("SQLAlchemy available: True")
print("Active adapter: SQLAlchemyAdapter")

## 3. SQLite PRAGMA Optimization

`optimize_pragma=True` temporarily changes PRAGMAs and restores captured settings after the operation. The selected tier depends on expected rows:

| Tier | Rows | Settings |
|---|---|---|
| Light | ≤ 10,000 | synchronous=NORMAL, temp_store=MEMORY, cache_size=-8000 |
| Moderate | 10,001–100,000 | synchronous=OFF, journal_mode=MEMORY, cache_size=-16000, mmap_size=256 MiB |
| Aggressive | > 100,000 | synchronous=OFF, journal_mode=OFF, cache_size=-32000, mmap_size=512 MiB; page_size=4096 |

Page-size changes may not apply to an existing database. The comparison below measures the actual small fixture run; it is not a performance guarantee.

In [ ]:

# Pragma optimization auto-tunes journal_mode, synchronous, etc. during batch writes
# Effect is more visible with large data volumes; small volumes are affected by UNIQUE constraint solving overhead
result_no_opt = check_result(fill(str(db_path), table="tasks", count=3000, optimize_pragma=False, clear_before=True), 3000)
print(f"Without optimization: {result_no_opt.count} rows in {result_no_opt.elapsed:.3f}s ({result_no_opt.rows_per_second:.0f} rows/s)")  # noqa: E501

result = check_result(fill(str(db_path), table="tasks", count=3000, optimize_pragma=True, clear_before=True), 3000)
print(f"With Pragma optimization: {result.count} rows in {result.elapsed:.3f}s ({result.rows_per_second:.0f} rows/s)")

## 4. Declared and Explicitly Configured Relations

Declared `FOREIGN KEY` constraints, such as `tasks.project_id → projects.project_id`, reuse existing parent keys. `reviews.member_id` has no declared FK. We configure that relation explicitly below; a shared column name alone is not a guarantee.

In [ ]:
import sqlite3

conn = sqlite3.connect(str(db_path))

print("--- reviews table FK ---")
fk_list = conn.execute("PRAGMA foreign_key_list(reviews)").fetchall()
print(f"Explicit FKs: {fk_list}")
print("member_id has no declared FK; the next cell configures an explicit association.")

conn.close()

## 5. SharedPool Cross-table Value Sharing

When multiple tables reference the same parent table, `SharedPool` ensures reference consistency:

- `tasks.project_id` references `projects.project_id` pool (explicit FK)
- `tasks.assignee_id` references `members.member_id` pool (explicit FK)
- `reviews.task_id` references `tasks.task_id` pool (explicit FK)
- `reviews.member_id` references `members.member_id` pool (explicit ColumnAssociation in the next cell)

See architecture.md §5 SharedPool

In [ ]:

with connect(str(db_path)) as orch:
    r1 = check_result(orch.fill_table("organizations", count=3), 3)
    r2 = check_result(orch.fill_table("members", count=10), 10)
    r3 = check_result(orch.fill_table("projects", count=5), 5)
    r4 = check_result(orch.fill_table("tasks", count=20), 20)


conn = sqlite3.connect(str(db_path))
project_ids_tasks = {r[0] for r in conn.execute("SELECT DISTINCT project_id FROM tasks").fetchall()}
project_ids_projects = {r[0] for r in conn.execute("SELECT project_id FROM projects").fetchall()}
print(f"project_ids in tasks: {project_ids_tasks}")
print(f"project_ids in projects: {project_ids_projects}")
print(f"All task project_ids exist in projects: {project_ids_tasks.issubset(project_ids_projects)}")
conn.close()
from sqlseed.config.models import ColumnAssociation
review_config = GeneratorConfig(
    db_path=str(db_path),
    tables=[TableConfig(name="reviews", count=10)],
    associations=[ColumnAssociation(column_name="member_id", source_table="members", source_column="member_id", target_tables=["reviews"])],
)
review_path = work_dir / "reviews.yaml"
save_config(review_config, str(review_path))
review_results = check_results(fill_from_config(str(review_path)), str(review_path))
with sqlite3.connect(str(db_path)) as connection:
    missing_members = connection.execute("SELECT count(*) FROM reviews WHERE member_id NOT IN (SELECT member_id FROM members)").fetchone()[0]
    assert missing_members == 0
    print("Unmatched review member IDs:", missing_members)


## 6. ColumnAssociation Explicit Association

For related columns without a declared FK, use the actual association fields:

```yaml
associations:
  - column_name: member_id
    source_table: members
    source_column: member_id
    target_tables: [reviews]
    strategy: shared_pool
```

The previous cell executed this association against the populated `members` table. See [06-config-deep-dive.ipynb](06-config-deep-dive.ipynb) for full configurations.

## 7. Multi-table Batch Filling

Use `fill_from_config` to batch-fill multiple tables from YAML/JSON config, auto-ordered by FK dependencies:

In [ ]:
from sqlseed import GeneratorConfig, ProviderType, TableConfig, fill_from_config
from sqlseed.config.loader import save_config

config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType("mimesis"),
    tables=[
        TableConfig(name="organizations", count=3),
        TableConfig(name="members", count=10),
        TableConfig(name="projects", count=5),
        TableConfig(name="tasks", count=30),
    ],
)

config_path = str((work_dir / "batch_config.yaml"))
save_config(config, config_path)

results = check_results(fill_from_config(config_path), config_path)
for r in results:
    print(f"{r.table_name}: {r.count} rows in {r.elapsed:.3f}s")

## 8. clear_before and FK Constraints

`clear_before=True` clears one table before filling it. For a graph reset, clear child tables before parent tables, or use a fresh database. `fill_from_config` orders generation by parent dependencies; it does not perform a separate children-first cleanup pass or make the whole graph atomic. These multi-table examples append to the temporary fixture.

## 9. BLOB Column Handling

BLOB-typed columns generate random binary data:

In [ ]:

# file_name matches *_name pattern -> name generator (generates person names)
# Here we override with columns for a more sensible filename format
rows = preview(str(db_path), table="attachments", count=2,
               columns={"file_name": {"type": "pattern", "regex": "[a-z]{8}[.](pdf|png|docx)"}})
for row in rows:
    print(f"file_name={row['file_name']}, uploaded_at={row['uploaded_at']}")
print("\nfile_data (BLOB, nullable) and file_size (DEFAULT 0) are skipped by the strategy chain")
print("BLOB columns are skipped by default in nullable strategy; columns with DEFAULT use the default value")

## 📋 DatabaseAdapter Full API

DatabaseAdapter Protocol defines all database operation interfaces:

In [ ]:
with sqlseed.connect(str(db_path)) as orch:
    print('=== get_table_names() ===')
    tables = orch.get_table_names()
    print(f'  Tables: {tables}')

    print('\n=== get_column_info(organizations) ===')
    col_info = orch.get_column_info('organizations')
    for col in col_info:
        print(f'  {col.name}: type={col.type}, pk={col.is_primary_key}')

    print('\n=== get_foreign_keys(tasks) ===')
    fks = orch.get_foreign_keys('tasks')
    for fk in fks:
        print(f'  {fk}')

    print('\n=== get_row_count() ===')
    for table in tables:
        count = orch.get_row_count(table)
        print(f'  {table}: {count} rows')

## ⚡ PRAGMA Tier Boundaries

Light applies through 10,000 expected rows, Moderate through 100,000, and Aggressive above 100,000. The table in section 3 lists the actual settings.

In [ ]:

print("PragmaOptimizer three-level optimization parameters:")
print("  Light threshold: <= 10,000 rows")
print("  Moderate threshold: 10,001 - 100,000 rows")
print("  Aggressive threshold: > 100,000 rows")

with sqlseed.connect(str(db_path), optimize_pragma=True) as orch:
    result = check_result(orch.fill_table("organizations", count=5), 5)
    print(f"\nFilled {result.count} rows (Light level)")

## 10. Summary

| Feature | Description |
|------|------|
| Dual adapter | SQLAlchemyAdapter (default) / RawSQLiteAdapter (test-only) |
| Pragma optimization | Three-level strategy, auto-selected |
| Explicit FK | Declared in CREATE TABLE |
| Undeclared relation | Configure ColumnAssociation explicitly |
| SharedPool | Cross-table reference consistency |
| ColumnAssociation | Explicit association declaration |
| fill_from_config | Auto topological sort, batch fill |
| BLOB | Random binary data |

**Next**: [05-dag-and-constraints.ipynb](05-dag-and-constraints.ipynb) — DAG topological sort and constraint solving

In [ ]:
# Verify exact database totals after the full notebook, plus every declared FK.
import sqlite3
expected_counts = {'organizations': 19, 'members': 50, 'projects': 25, 'tasks': 3050, 'tags': 13, 'reviews': 10}
with sqlite3.connect(str(db_path)) as verification_db:
    actual_counts = {
        table: verification_db.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
        for table in expected_counts  # Fixed tutorial table names.
    }
    assert actual_counts == expected_counts, (actual_counts, expected_counts)
    fk_errors = verification_db.execute("PRAGMA foreign_key_check").fetchall()
    assert not fk_errors, fk_errors
print("Verified database row counts:", actual_counts)
print("Database FK check:", fk_errors)
print("Verified fill operations:", len(generation_checks))
